# 🎨 Image Restoration Dual-Model Training - Colab

This notebook supports **Progressive Training** for both **SwinIR-Light** (Deterministic) and **Stable Diffusion LoRA** (Generative).
Ensure you have uploaded `restoration_training.zip` to your Google Drive.

### 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Create persistent weights directory if it doesn't exist
!mkdir -p /content/drive/MyDrive/weights/sd_lora_antique_colab
!mkdir -p /content/drive/MyDrive/weights/swinir_checkpoints

### 2. Extract Project & Setup Environment

In [ ]:
# Unzip the project folder
!unzip -q /content/drive/MyDrive/restoration_training.zip -d /content/

%cd /content/training

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q xformers

### 3. Data Preparation (Optional)
Run this if you uploaded raw images to `/content/training/datasets/raw` and need to generate the clean/damaged pairs.

In [ ]:
# Example: Generating processed data from raw images
# !python scripts/prepare_data.py --input ./datasets/raw --output ./datasets/processed --seed 42

### 4. Track 1: SwinIR-Light Training
Uses `--resume` to automatically continue from the latest checkpoint if training was interrupted.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/clean \
                                --damaged_dir ./datasets/processed/damaged \
                                --epochs 100 \
                                --batch_size 16 \
                                --patch_size 128 \
                                --lr 1e-4 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                --resume

### 5. Track 2: Stable Diffusion LoRA Transfer Training
Uses `accelerator.save_state()` to preserve full training context (optimizer, scheduler, etc).

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/clean \
                                 --epochs 20 \
                                 --batch_size 4 \
                                 --lr 1e-4 \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_colab \
                                 --resume